# Mini-TP 7 · Seguridad con SAIF — REST (FastAPI)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este es **uno de los seis notebooks guiados** del Mini-TP 7. Cada uno asegura **una capa**
de las que construimos en el bimestre. **Elige este si tu modelo se expone por REST (FastAPI)**
(o simplemente para practicar esta estrategia). Con completar **uno** alcanza para la entrega.

### SAIF en esta capa
**Riesgo:** abuso, inyección, DoS y *scraping* para robar el modelo.
**Controles:** autenticación (API key/JWT) · validación de entrada · *rate limiting*.
**Elementos SAIF:** bases sólidas · automatizar defensas.

### Qué vas a hacer
1. Ver la **versión vulnerable** y por qué es un riesgo.
2. Aplicar el **control principal** (guiado, ya resuelto como ejemplo).
3. Completar los **`# TODO`** sobre tu propio modelo.
4. Escribir una **reflexión** (amenaza mitigada ↔ elemento de SAIF).

> Corre de punta a punta con **uv**, sin servicios externos.

### Dependencias
```bash
uv add fastapi httpx pyjwt pydantic scikit-learn numpy pandas xgboost
```

## 0. Mi modelo
Igual que en los Mini-TP 1 a 6, uso **mi propio modelo**: el XGBoost **`xgb_best`** del TP
integrador, que predice el **precio de autos usados** (dataset Car Dekho).

Como este notebook tiene que correr **sin servicios externos**, no lo cargo desde MLflow/MinIO:
lo entreno acá con los mismos pasos e hiperparámetros del DAG de Airflow, reusando
`MLOps1_final/common` **sin copiarlo** (igual que en el Mini-TP 6). También reuso el esquema
`CarRawInput`, que es el que valida la entrada en la API REST del TP integrador.

A diferencia del modelo de ejemplo (30 números ya normalizados), mi API recibe **un auto con
datos crudos** (marca, año, km, combustible, "23.4 kmpl", "1248 CC", ...). Eso cambia bastante
qué significa "validar la entrada" (ver TODO 3).

In [1]:
import os, sys, time, json, hashlib, datetime, secrets
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

# Reusar el preprocesamiento del TP integrador (MLOps1_final/common)
TP_MLOPS1 = os.path.abspath(os.path.join("..", "TP_Final", "MLOps1_final"))
if TP_MLOPS1 not in sys.path:
    sys.path.insert(0, TP_MLOPS1)
DATASET_CSV = os.path.join(TP_MLOPS1, "dataset", "Car details v3.csv")

from common.constants import XGB_PARAMS, TRAIN_TEST_SIZE, TRAIN_TEST_SPLIT_SEED, MLFLOW_MODEL_NAME
from common.preprocessing import (
    extraer_columnas_tecnicas, drop_missing_technical_rows, drop_km_outliers,
    compute_top_brands, encode_categoricals, get_all_feature_columns,
    align_columns, build_feature_matrix, build_inference_features,
)
from common.schemas import CarRawInput

# Dataset y split estratificado por rango de precio (igual que en el TP integrador)
df = pd.read_csv(DATASET_CSV).drop_duplicates().reset_index(drop=True)
price_bins = pd.qcut(df["selling_price"], q=5,
                     labels=["muy_barato", "barato", "intermedio", "caro", "muy_caro"])
df_train, df_test = train_test_split(df, test_size=TRAIN_TEST_SIZE,
                                     stratify=price_bins, random_state=TRAIN_TEST_SPLIT_SEED)

# Feature engineering
df_train = drop_km_outliers(drop_missing_technical_rows(extraer_columnas_tecnicas(df_train)))
df_test = drop_missing_technical_rows(extraer_columnas_tecnicas(df_test))
top_brands = compute_top_brands(df_train)
df_train_enc, df_test_enc = encode_categoricals(df_train, df_test, top_brands)
all_features = get_all_feature_columns(df_train_enc)
df_train_enc, df_test_enc = align_columns(df_train_enc, df_test_enc, all_features)

X_train = build_feature_matrix(df_train_enc, all_features)
X_test = build_feature_matrix(df_test_enc, all_features)
y_train = np.log1p(df_train_enc["selling_price"])   # el modelo predice log(precio)

modelo = XGBRegressor(**XGB_PARAMS)
modelo.fit(X_train, y_train)

pred_test = np.expm1(modelo.predict(X_test))
print(f"Modelo {MLFLOW_MODEL_NAME} listo. R2 test = {r2_score(df_test_enc['selling_price'], pred_test):.4f}"
      f" | MAE test = {mean_absolute_error(df_test_enc['selling_price'], pred_test):,.0f}"
      f" | n_features = {len(all_features)}")

def predecir_precio(car: CarRawInput) -> float:
    """Mismo camino que la API del TP integrador: auto crudo -> features -> precio."""
    fila = build_inference_features(car, top_brands, all_features)
    return float(np.expm1(modelo.predict(fila)[0]))

# Auto de ejemplo (primer auto del dataset, precio real 450.000)
AUTO = {
    "name": "Maruti Swift Dzire VDI", "year": 2014, "km_driven": 145500,
    "fuel": "Diesel", "seller_type": "Individual", "transmission": "Manual",
    "owner": "First Owner", "mileage": "23.4 kmpl", "engine": "1248 CC",
    "max_power": "74 bhp", "torque": "190Nm@ 2000rpm", "seats": 5,
}
print(f"Precio predicho para el auto de ejemplo: {predecir_precio(CarRawInput(**AUTO)):,.0f}")

Modelo xgb_best listo. R2 test = 0.9372 | MAE test = 72,408 | n_features = 23
Precio predicho para el auto de ejemplo: 455,294


## 1. Versión VULNERABLE — cualquiera consulta el modelo, sin límites
El endpoint recibe un `dict` cualquiera: no pide credenciales, no valida nada y no limita
cuántas consultas se hacen.

In [2]:
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

from types import SimpleNamespace

def sin_validar(payload: dict):
    """Arma el auto tal cual llega, SIN pasar por Pydantic (lo que hace una API descuidada)."""
    enums = {"fuel", "seller_type", "transmission", "owner"}
    return SimpleNamespace(**{k: SimpleNamespace(value=v) if k in enums else v for k, v in payload.items()})

app_v = FastAPI()
@app_v.post("/predict")
def predict_v(payload: dict):
    fila = build_inference_features(sin_validar(payload), top_brands, all_features)
    return {"precio": float(np.expm1(modelo.predict(fila)[0]))}

# raise_server_exceptions=False: ver el 500 como lo vería un cliente, en vez de la excepción
cli_v = TestClient(app_v, raise_server_exceptions=False)

# a) Sin credenciales: cualquiera consulta
print("sin auth           :", cli_v.post("/predict", json=AUTO).status_code)

# b) Basura en la entrada: rompe el servidor (500) o, peor, devuelve un precio inventado
print("falta un campo     :", cli_v.post("/predict", json={"name": "x"}).status_code, "(500: error interno expuesto)")
basura = {**AUTO, "mileage": "mucho", "engine": "<script>", "km_driven": -5, "year": 3000}
r = cli_v.post("/predict", json=basura)
print("datos absurdos     :", r.status_code, "->", r.json(), "(el modelo predice igual)")

# c) Scraping: 300 consultas seguidas variando una feature = curva de precio del modelo
t0 = time.time()
curva = [cli_v.post("/predict", json={**AUTO, "km_driven": km}).json()["precio"]
         for km in range(0, 300_000, 1_000)]
print(f"300 consultas      : todas permitidas en {time.time()-t0:.1f}s (riesgo de extracción del modelo)")

sin auth           : 200
falta un campo     : 500 (500: error interno expuesto)
datos absurdos     : 200 -> {'precio': 802850.75} (el modelo predice igual)
300 consultas      : todas permitidas en 8.3s (riesgo de extracción del modelo)


## 2. Control principal (GUIADO) — auth + validación + rate limiting
El mismo control del ejemplo, pero sobre **mi modelo**. Le hice tres ajustes al código guiado:

- **Una key por cliente** y **rate limit por cliente**. En el ejemplo `auth()` devolvía siempre
  `"cliente"`, así que todos compartían el mismo cupo de 5/min (un cliente abusivo bloqueaba a los demás).
- La key se compara con **`secrets.compare_digest`** (tiempo constante, evita ataques de *timing*).
- Las keys salen de una **variable de entorno** (con un valor de demo si no está definida).

La validación usa `CarRawInput`, el esquema de la API REST del TP integrador.

In [3]:
from fastapi import Depends, HTTPException
from fastapi.security import APIKeyHeader

# En producción: export TP7_API_KEYS='{"key-...": "cliente", ...}' (nunca en el código)
API_KEYS = json.loads(os.environ.get(
    "TP7_API_KEYS", '{"demo-key-web": "app-web", "demo-key-partner": "partner"}'))
LIMITE_POR_MINUTO = 5

api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)
def auth(k: str = Depends(api_key_header)):
    for key, cliente in API_KEYS.items():
        if k and secrets.compare_digest(k, key):
            return cliente
    raise HTTPException(401, "no autorizado")

_hits = {}
def rate_limit(cliente: str = Depends(auth)):
    now = time.time(); w = _hits.setdefault(cliente, [])
    w[:] = [t for t in w if now - t < 60]          # ventana deslizante de 60 s
    if len(w) >= LIMITE_POR_MINUTO:
        raise HTTPException(429, "demasiadas solicitudes")
    w.append(now); return cliente

app = FastAPI()
@app.post("/predict")
def predict(x: CarRawInput, cliente: str = Depends(rate_limit)):
    return {"precio": round(predecir_precio(x))}

cli = TestClient(app)
h_web, h_partner = {"X-API-Key": "demo-key-web"}, {"X-API-Key": "demo-key-partner"}
print("sin key       :", cli.post("/predict", json=AUTO).status_code, "(401)")
print("key inventada :", cli.post("/predict", json=AUTO, headers={"X-API-Key": "hola"}).status_code, "(401)")
print("con key       :", cli.post("/predict", json=AUTO, headers=h_web).status_code, "(200)")
print("mala forma    :", cli.post("/predict", json={"name": "x"}, headers=h_web).status_code, "(422)")
print("fuel inválido :", cli.post("/predict", json={**AUTO, "fuel": "Nafta"}, headers=h_web).status_code, "(422)")
codes = [cli.post("/predict", json=AUTO, headers=h_web).status_code for _ in range(5)]
print("rate limit    :", codes, "(429 al superar 5/min)")
print("otro cliente  :", cli.post("/predict", json=AUTO, headers=h_partner).status_code,
      "(200: su cupo es independiente)")

sin key       : 401 (401)
key inventada : 401 (401)
con key       : 200 (200)
mala forma    : 422 (422)
fuel inválido : 422 (422)
rate limit    : [200, 200, 429, 429, 429] (429 al superar 5/min)
otro cliente  : 200 (200: su cupo es independiente)


**Por qué el rate limit corta antes de 5 respuestas 200:** FastAPI resuelve las dependencias
(`auth` → `rate_limit`) **antes** de validar el body. Entonces las consultas con entrada
inválida (422) **también consumen cupo**. Está bien que sea así: un atacante que manda basura
también está abusando del servicio.

**Lo que `CarRawInput` todavía deja pasar:** valida los enums (fuel, owner, ...), el año y los
asientos, pero `mileage`, `engine`, `max_power` y `torque` son **texto libre**:

In [4]:
_hits.clear()
r = cli.post("/predict", json={**AUTO, "mileage": "mucho", "engine": "<script>",
                               "km_driven": 50_000_000}, headers=h_web)
print(r.status_code, r.json(), "<- entrada sin sentido, pero el modelo devuelve un precio igual")

200 {'precio': 398369} <- entrada sin sentido, pero el modelo devuelve un precio igual


## 3. Tu entrega — completa los `# TODO`
Sobre **tu propio modelo**: (a) agrega **autorización por rol con JWT**, y (b) un
**logging de auditoría** (medida de gobernanza).

Empiezo por el **TODO 3** (la entrada), porque los otros dos lo usan.

### TODO 3 — `Entrada` con las features REALES de mi modelo
Mi entrada no son 30 números, es un auto crudo. Entonces la valido en dos niveles:

1. **Campos** con `Field(ge=, le=)` y `pattern`: año, km, asientos, y el formato de los textos
   técnicos ("23.4 kmpl", "1248 CC", "74 bhp"). `extra="forbid"` rechaza campos de más.
2. **Valores derivados**: paso el auto por el **mismo** `extraer_columnas_tecnicas` del
   entrenamiento y exijo que mileage, motor, potencia y torque **se puedan leer** y queden
   **dentro del rango visto en train** (con un margen del 20 %). Así el modelo nunca predice
   sobre un auto que no se parece a nada de lo que aprendió.

Los rangos **no están escritos a mano**: salen de `df_train`.

In [5]:
from pydantic import Field, ConfigDict, model_validator
from common.constants import KM_OUTLIER_THRESHOLD, NULL_CHECK_COLUMNS, REFERENCE_YEAR

# Rangos reales de las features técnicas, sacados de train (+-20 % de margen)
COLS_TECNICAS = ["mileage_value", "engine_value", "max_power_value", "torque_nm"]
RANGOS = {c: (float(df_train[c].min()) * 0.8, float(df_train[c].max()) * 1.2) for c in COLS_TECNICAS}
ANIO_MIN = int(df_train["year"].min())
for c, (lo, hi) in RANGOS.items():
    print(f"  {c:16s} [{lo:8.1f}, {hi:8.1f}]")
print(f"  {'year':16s} [{ANIO_MIN}, {REFERENCE_YEAR + 6}]")

NUM = r"^\d+(\.\d+)?"
class Entrada(CarRawInput):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    name: str = Field(..., min_length=3, max_length=80, pattern=r"^[A-Za-z0-9 .\-()/+&']+$")
    year: int = Field(..., ge=ANIO_MIN, le=REFERENCE_YEAR + 6)
    km_driven: int = Field(..., ge=0, le=KM_OUTLIER_THRESHOLD)
    seats: float = Field(..., ge=2, le=14)
    mileage: str = Field(..., max_length=20, pattern=NUM + r" ?(kmpl|km/kg)$")
    engine: str = Field(..., max_length=20, pattern=NUM + r" ?CC$")
    max_power: str = Field(..., max_length=20, pattern=NUM + r" ?bhp$")
    torque: str = Field(..., max_length=40, pattern=r"^\d")

    @model_validator(mode="after")
    def features_en_rango(self):
        fila = extraer_columnas_tecnicas(pd.DataFrame([self.model_dump(mode="json")])).iloc[0]
        faltan = [c for c in NULL_CHECK_COLUMNS if pd.isna(fila[c])]
        if faltan:
            raise ValueError(f"no se pudieron leer: {faltan}")
        for c, (lo, hi) in RANGOS.items():
            if not lo <= fila[c] <= hi:
                raise ValueError(f"{c}={fila[c]:.1f} fuera del rango de entrenamiento [{lo:.1f}, {hi:.1f}]")
        return self

# Prueba rápida del esquema
from pydantic import ValidationError
casos = {
    "auto válido": AUTO,
    "mileage basura": {**AUTO, "mileage": "mucho"},
    "km imposibles": {**AUTO, "km_driven": 50_000_000},
    "motor de 90.000 CC": {**AUTO, "engine": "90000 CC"},
    "campo extra": {**AUTO, "es_admin": True},
    "torque de 9 Nm": {**AUTO, "torque": "9Nm@ 2000rpm"},
}
for nombre, datos in casos.items():
    try:
        Entrada(**datos); print(f"  OK       {nombre}")
    except ValidationError as e:
        print(f"  RECHAZO  {nombre:20s} -> {e.errors()[0]['msg'][:75]}")

# El esquema no debe rechazar autos reales: lo pruebo con todo el set de test
validos = 0
for row in df_test[list(AUTO)].to_dict("records"):
    try:
        Entrada(**row); validos += 1
    except ValidationError:
        pass
print(f"Autos de test aceptados: {validos}/{len(df_test)} ({validos/len(df_test):.1%})")

  mileage_value    [     0.0,     50.4]
  engine_value     [   499.2,   4324.8]
  max_power_value  [    26.2,    480.0]
  torque_nm        [    37.7,   4471.8]
  year             [1994, 2026]
  OK       auto válido
  RECHAZO  mileage basura       -> String should match pattern '^\d+(\.\d+)? ?(kmpl|km/kg)$'
  RECHAZO  km imposibles        -> Input should be less than or equal to 1000000
  RECHAZO  motor de 90.000 CC   -> Value error, engine_value=90000.0 fuera del rango de entrenamiento [499.2, 
  RECHAZO  campo extra          -> Extra inputs are not permitted
  RECHAZO  torque de 9 Nm       -> Value error, torque_nm=9.0 fuera del rango de entrenamiento [37.7, 4471.8]
Autos de test aceptados: 1683/1683 (100.0%)


### TODO 1 — autorización por rol con JWT
- El token Bearer se lee con `HTTPBearer` y se verifica con `jwt.decode(..., algorithms=["HS256"])`.
  Fijar el algoritmo es importante: sin eso, un atacante podría mandar un token con `alg: none`
  (sin firma) y que se acepte.
- **401** si no hay token, está vencido o la firma no coincide. **403** si el token es válido
  pero el rol no está en `{"analista", "admin"}`.
- `HTTPBearer(auto_error=False)`: si no, FastAPI responde 403 cuando falta el token, y lo
  correcto es 401 (no autenticado ≠ no autorizado).
- `exp` con `datetime.now(timezone.utc)`, porque `utcnow()` está deprecado.

### TODO 2 — logging de auditoría (gobernanza)
Cada predicción queda registrada con: fecha y hora, usuario, rol, **hash SHA-256 de la entrada**
(no guardo los datos crudos, así el log no se vuelve otra fuente de fuga), precio predicho y
versión del modelo. También registro los **accesos denegados**, que es lo que más interesa
revisar en una auditoría.

In [6]:
import jwt   # uv add pyjwt
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials

# En producción: secreto desde variable de entorno, nunca en el código
JWT_SECRET = os.environ.get("TP7_JWT_SECRET", "cambiar-en-produccion-con-32-bytes-o-mas")
ROLES_PERMITIDOS = {"analista", "admin"}
MODEL_VERSION = f"{MLFLOW_MODEL_NAME}-local"
AUDITORIA = []

def ahora_utc():
    return datetime.datetime.now(datetime.timezone.utc)

def crear_token(usuario, rol, minutos=15, secreto=JWT_SECRET):
    payload = {"sub": usuario, "rol": rol, "exp": ahora_utc() + datetime.timedelta(minutes=minutos)}
    return jwt.encode(payload, secreto, algorithm="HS256")

def auditar(evento, usuario, rol=None, entrada=None, precio=None, detalle=None):
    AUDITORIA.append({
        "ts": ahora_utc().isoformat(timespec="seconds"),
        "evento": evento, "usuario": usuario, "rol": rol,
        "hash_entrada": (hashlib.sha256(json.dumps(entrada, sort_keys=True).encode()).hexdigest()[:16]
                         if entrada else None),
        "precio": precio, "modelo": MODEL_VERSION if precio else None, "detalle": detalle,
    })

# TODO 1: dependencia requiere_rol()
bearer = HTTPBearer(auto_error=False)
def requiere_rol(cred: HTTPAuthorizationCredentials = Depends(bearer)):
    if cred is None:
        auditar("denegado", None, detalle="sin token")
        raise HTTPException(401, "falta el token", headers={"WWW-Authenticate": "Bearer"})
    try:
        datos = jwt.decode(cred.credentials, JWT_SECRET, algorithms=["HS256"],
                           options={"require": ["exp", "sub"]})
    except jwt.ExpiredSignatureError:
        auditar("denegado", None, detalle="token vencido")
        raise HTTPException(401, "token vencido", headers={"WWW-Authenticate": "Bearer"})
    except jwt.InvalidTokenError as e:
        auditar("denegado", None, detalle=f"token inválido: {type(e).__name__}")
        raise HTTPException(401, "token inválido", headers={"WWW-Authenticate": "Bearer"})
    if datos.get("rol") not in ROLES_PERMITIDOS:
        auditar("denegado", datos["sub"], datos.get("rol"), detalle="rol sin permiso")
        raise HTTPException(403, "rol sin permiso para predecir")
    return {"usuario": datos["sub"], "rol": datos["rol"]}

# Rate limit por usuario del token (no por un cliente fijo)
_hits_jwt = {}
def rate_limit_jwt(u: dict = Depends(requiere_rol)):
    now = time.time(); w = _hits_jwt.setdefault(u["usuario"], [])
    w[:] = [t for t in w if now - t < 60]
    if len(w) >= LIMITE_POR_MINUTO:
        auditar("limitado", u["usuario"], u["rol"], detalle="rate limit")
        raise HTTPException(429, "demasiadas solicitudes")
    w.append(now); return u

app_seg = FastAPI()
@app_seg.post("/predict")
def predict_seguro(x: Entrada, u: dict = Depends(rate_limit_jwt)):
    precio = round(predecir_precio(x))
    # TODO 2: registro de auditoría de cada predicción
    auditar("prediccion", u["usuario"], u["rol"], entrada=x.model_dump(mode="json"), precio=precio)
    return {"precio": precio, "modelo": MODEL_VERSION}

print("Token de ejemplo:", crear_token("jaime", "analista")[:30], "...")

Token de ejemplo: eyJhbGciOiJIUzI1NiIsInR5cCI6Ik ...


### Pruebas de punta a punta

In [7]:
cli_s = TestClient(app_seg)
def bearer_h(token): return {"Authorization": f"Bearer {token}"}
def post(json_=AUTO, token=None):
    return cli_s.post("/predict", json=json_, headers=bearer_h(token) if token else {})

t_ana, t_admin = crear_token("jaime", "analista"), crear_token("ana", "admin")
t_inv = crear_token("pepe", "invitado")
t_vencido = crear_token("jaime", "analista", minutos=-1)
t_falso = crear_token("jaime", "admin", secreto="otro-secreto-que-no-es-el-del-servidor!!")
t_none = jwt.encode({"sub": "hacker", "rol": "admin", "exp": ahora_utc() + datetime.timedelta(minutes=5)},
                    None, algorithm="none")    # token SIN firma

pruebas = [
    ("sin token",                    post(),                         401),
    ("token vencido",                post(token=t_vencido),          401),
    ("firmado con otro secreto",     post(token=t_falso),            401),
    ("alg=none (sin firma)",         post(token=t_none),             401),
    ("rol invitado",                 post(token=t_inv),              403),
    ("analista",                     post(token=t_ana),              200),
    ("admin",                        post(token=t_admin),            200),
    ("mileage basura (analista)",    post({**AUTO, "mileage": "mucho"}, t_ana), 422),
    ("campo extra (analista)",       post({**AUTO, "rol": "admin"}, t_ana),     422),
]
ok = True
for nombre, r, esperado in pruebas:
    ok &= r.status_code == esperado
    print(f"  {nombre:28s} {r.status_code}  (esperado {esperado})  {'OK' if r.status_code == esperado else 'FALLA'}")

# jaime ya usó 3 de sus 5 consultas del minuto (200 + dos 422)
codes = [post(token=t_ana).status_code for _ in range(4)]
print(f"  {'rate limit jaime':28s} {codes}  (2 x 200 y después 429)")
ok &= codes == [200, 200, 429, 429]
print(f"  {'ana sigue con cupo propio':28s} {post(token=t_admin).status_code}")
print("\nTodas las pruebas OK" if ok else "\nHAY PRUEBAS QUE FALLARON")

  sin token                    401  (esperado 401)  OK
  token vencido                401  (esperado 401)  OK
  firmado con otro secreto     401  (esperado 401)  OK
  alg=none (sin firma)         401  (esperado 401)  OK
  rol invitado                 403  (esperado 403)  OK
  analista                     200  (esperado 200)  OK
  admin                        200  (esperado 200)  OK
  mileage basura (analista)    422  (esperado 422)  OK
  campo extra (analista)       422  (esperado 422)  OK
  rate limit jaime             [200, 200, 429, 429]  (2 x 200 y después 429)
  ana sigue con cupo propio    200

Todas las pruebas OK


In [8]:
# Registro de auditoría (en producción iría a un log append-only / data lake, no a una lista)
pd.set_option("display.width", 160)
pd.DataFrame(AUDITORIA)

,ts,evento,usuario,rol,hash_entrada,precio,modelo,detalle
0,2026-10-08T23:37:45+00:00,denegado,None,None,None,NaN,None,sin token
1,2026-10-08T23:37:45+00:00,denegado,None,None,None,NaN,None,token vencido
2,2026-10-08T23:37:45+00:00,denegado,None,None,None,NaN,None,token inválido: InvalidSignatureError
3,2026-10-08T23:37:45+00:00,denegado,None,None,None,NaN,None,token inválido: InvalidAlgorithmError
4,2026-10-08T23:37:45+00:00,denegado,pepe,invitado,None,NaN,None,rol sin permiso
5,2026-10-08T23:37:45+00:00,prediccion,jaime,analista,75d4e8e2c1da1592,455294.0,xgb_best-local,None
6,2026-10-08T23:37:45+00:00,prediccion,ana,admin,75d4e8e2c1da1592,455294.0,xgb_best-local,None
7,2026-10-08T23:37:45+00:00,prediccion,jaime,analista,75d4e8e2c1da1592,455294.0,xgb_best-local,None
8,2026-10-08T23:37:45+00:00,prediccion,jaime,analista,75d4e8e2c1da1592,455294.0,xgb_best-local,None
9,2026-10-08T23:37:45+00:00,limitado,jaime,analista,None,NaN,None,rate limit


## 4. Reflexión

**1. ¿Qué amenaza frena el rate limiting y por qué se relaciona con el robo del modelo?**
Frena el **abuso** del servicio (DoS, costo) y sobre todo la **extracción del modelo**. En la
versión vulnerable hice 300 consultas seguidas variando solo los km y obtuve la curva de precio
del modelo. Con suficientes consultas así, alguien puede entrenar una copia de mi XGBoost
(un modelo "sustituto") sin haber tenido nunca el dataset ni el artefacto. Con un límite de
consultas por usuario, ese ataque se vuelve lento y caro, y como cada consulta queda
**auditada con su usuario**, también se vuelve visible. Para dificultarlo más, la API devuelve
el precio **redondeado** y no expone nada más (ni features, ni probabilidades, ni versión interna
del árbol).

**2. ¿Qué elemento de SAIF cubre mi control principal?**

| Control | Amenaza | Elemento de SAIF |
|---|---|---|
| API key / JWT con rol (401/403) | acceso no autorizado | **Bases sólidas de seguridad** (*strong security foundations*) |
| Validación con `Entrada` (422) | inyección, entradas fuera de dominio | **Bases sólidas de seguridad** |
| Rate limiting por usuario (429) | abuso, DoS, extracción del modelo | **Automatizar defensas** (*automate defenses*) |
| Auditoría (hash de entrada, usuario, precio) | uso indebido sin rastro | **Detección y respuesta** (*extend detection and response*) + gobernanza |
| Secretos fuera del código, `compare_digest`, `alg` fijo | robo de credenciales, tokens falsos | **Bases sólidas de seguridad** |

El control principal (auth + validación + rate limiting) cubre **bases sólidas** y
**automatizar defensas**: son controles clásicos de seguridad aplicados al endpoint de un modelo,
y funcionan solos, sin que nadie tenga que mirar.

**3. ¿Por qué validar la entrada es también una medida de seguridad y no solo de calidad?**
Porque el modelo **nunca se niega a responder**: en la versión vulnerable, un auto con
`mileage="mucho"`, `engine="<script>"`, año 3000 y km negativos devolvió **un precio**, como si
fuera un dato real. Validar la entrada:
- cierra la puerta a **inyección** y a cargas raras que rompen el servidor (los 500 de la versión
  vulnerable además exponían errores internos);
- impide que alguien **explore al modelo fuera de su dominio**, que es justo donde se buscan
  entradas adversarias y donde el modelo se comporta peor;
- protege lo que viene después: si las predicciones se guardan para reentrenar (como las del
  streaming de mi TP integrador), la basura que entra hoy sería **envenenamiento** mañana.

Por eso mi `Entrada` no solo revisa tipos: usa el **mismo preprocesamiento del entrenamiento**
y exige que el auto esté dentro de los rangos que el modelo vio en train.